# Marginal MAP over a subset of times

`viterbi_torch_mvr_chmm` answers "what is the single most likely hidden path?".
This notebook is about a different question: **what is the most likely
assignment at a few times of interest, once every other time is averaged out?**

The HMM × MVR product is itself an HMM, with state $z = (x, m)$ and with each
constraint enforced by conditioning on $\mathrm{evl}(m) = \text{True}$ at the end
of its window. The query is plain marginal MAP over that chain:

$$\arg\max_{z_S}\ \sum_{z_{\bar S}}\ P(z, y)$$

for a chosen query set $S$. So the mediation state is **maximized** at the query
times alongside the hidden state, and the hidden path reported below is the
projection of that augmented argmax.

This is *marginal MAP*, and it is **not** the Viterbi path restricted to $S$.
Maximizing a marginal and marginalizing a maximum are different operations, and
the notebook below shows a case where they disagree.

The times outside $S$ are summed out, not deleted: they still consume a
transition, still emit if they carry an observation, and still drive every MVR
active at that time. What is dropped is only the requirement to commit to a
value there.

In [ ]:
import itertools
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.inference.viterbi_mvr import viterbi_torch_mvr_chmm
from conin.hidden_markov_model.inference.marginal_map_mvr import (
    marginal_map_torch_mvr_chmm,
)

## 1. The model

The same three-state HMM used in `MVR_viterbi.ipynb`, so the two notebooks can be
read against each other.

In [ ]:
HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

start_probs = {
    "A": 0.2765440507007986,
    "B": 0.4033576072467887,
    "C": 0.32009834205241255,
}

transition_probs = {
    ("A", "A"): 0.3391777054270445,
    ("A", "B"): 0.049711711669595204,
    ("A", "C"): 0.6111105829033604,
    ("B", "A"): 0.48102507253852517,
    ("B", "B"): 0.05601918704283972,
    ("B", "C"): 0.4629557404186351,
    ("C", "A"): 0.43616112524444134,
    ("C", "B"): 0.1773076392327265,
    ("C", "C"): 0.38653123552283214,
}

emission_probs = {
    ("A", "lo"): 0.19949219710155375,
    ("A", "mid"): 0.30789837305397333,
    ("A", "hi"): 0.492609429844473,
    ("B", "lo"): 0.534907622618408,
    ("B", "mid"): 0.234417585356662,
    ("B", "hi"): 0.23067479202493,
    ("C", "lo"): 0.09093879934300991,
    ("C", "mid"): 0.008996844382398088,
    ("C", "hi"): 0.9000643562745919,
}

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs=start_probs,
    transition_probs=transition_probs,
    emission_probs=emission_probs,
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]
T = len(observed)

print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

`T = 7` is small enough that every one of the $3^7 = 2187$ hidden paths can be
enumerated, so every claim below is checkable by brute force.

## 2. A reference implementation

Marginal MAP is easy to state exhaustively: enumerate the paths, group them by
what they do at the query times, add up the probability inside each group, and
take the heaviest group. That is the definition, and it is what the fast
algorithm has to reproduce.

One caveat if you copy this reference elsewhere: it groups by **hidden state
alone**, which is only valid for the constraint used below. In general the group
key is the augmented state — hidden plus the mediation label of every MVR active
at that time. The two coincide here because `forbid`'s alternative mediation
value is the absorbing `violated` state, which the acceptance filter has already
removed, so exactly one mediation value survives per hidden state. A constraint
that keeps two values live at a query time — a parity counter, say — makes the
two groupings genuinely different queries, and this cell would then be answering
the other one.

In [ ]:
def path_logprob(path, obs=None):
    """Joint log probability of a hidden path and the observations."""
    obs = observed if obs is None else obs
    obs_map = obs if isinstance(obs, dict) else dict(enumerate(obs))
    repn = hmm.repn
    idx = [hmm.hidden_to_internal[h] for h in path]

    total = math.log(repn.start_vec[idx[0]])
    for t in range(1, len(idx)):
        total += math.log(repn.transition_mat[idx[t - 1]][idx[t]])
    for t, o in obs_map.items():
        total += math.log(repn.emission_mat[idx[t]][hmm.observed_to_internal[o]])

    return total


def group_masses(query_times, horizon=None, obs=None, accepts=None):
    """Total probability of every assignment to the query times."""
    horizon = T if horizon is None else horizon
    masses = {}

    for path in itertools.product(HIDDEN_STATES, repeat=horizon):
        if accepts is not None and not accepts(path):
            continue
        key = tuple(path[t] for t in query_times)
        masses[key] = masses.get(key, 0.0) + math.exp(path_logprob(path, obs))

    return masses


def brute_force_marginal_map(query_times, **kwargs):
    masses = group_masses(query_times, **kwargs)
    best = max(masses, key=masses.get)
    return list(best), math.log(masses[best])


print(f"{3 ** T} hidden paths to enumerate")

## 3. The headline: marginal MAP is not Viterbi restricted

Query the first, middle and last times $S = \{0, 3, 6\}$ and compare three
things: the marginal-MAP answer, the Viterbi path cut down to those times, and
the brute-force truth.

In [ ]:
QUERY = [0, 3, 6]

model = MVR_CHMM(hidden_markov_model=hmm, constraints=[])

viterbi_path, viterbi_ll = viterbi_torch_mvr_chmm(
    model, observed, return_augmented=False, return_score=True
)
restricted = [viterbi_path[t] for t in QUERY]

mm_path, mm_score = marginal_map_torch_mvr_chmm(
    model, observed, query_times=QUERY,
    return_augmented=False, return_score=True,
)

bf_path, bf_score = brute_force_marginal_map(QUERY)

print(f"full Viterbi path        : {' '.join(viterbi_path)}   ll = {viterbi_ll:.4f}")
print(f"  ... restricted to {QUERY} : {' '.join(restricted)}")
print()
print(f"marginal MAP at {QUERY}   : {' '.join(mm_path)}   log-mass = {mm_score:.4f}")
print(f"brute force               : {' '.join(bf_path)}   log-mass = {bf_score:.4f}")
print()
print("marginal MAP matches brute force :", mm_path == bf_path
      and abs(mm_score - bf_score) < 1e-4)
print("marginal MAP == restricted Viterbi:", mm_path == restricted)

The two disagree, and the marginal-MAP answer is the correct one *for the
question being asked*. The reason is visible in the next cell: the Viterbi
restriction names the single heaviest **path**, while marginal MAP names the
heaviest **group of paths**. A group made of many individually-mediocre paths
can outweigh a group built around one excellent one.

In [ ]:
masses = group_masses(QUERY)
total = sum(masses.values())
ranked = sorted(masses.items(), key=lambda kv: -kv[1])

best_in_group = {}
for p in itertools.product(HIDDEN_STATES, repeat=T):
    k = tuple(p[t] for t in QUERY)
    best_in_group[k] = max(best_in_group.get(k, -math.inf), path_logprob(p))

# Always show the Viterbi restriction, wherever it happens to rank.
shown = [k for k, _ in ranked[:8]]
if tuple(restricted) not in shown:
    shown.append(tuple(restricted))

table = pd.DataFrame(
    [
        {
            "rank": [k for k, _ in ranked].index(k) + 1,
            "assignment": " ".join(k),
            "posterior mass": masses[k] / total,
            "best single path in group": best_in_group[k],
        }
        for k in shown
    ]
)

def highlight(row):
    key = row["assignment"].split()
    if key == mm_path:
        return ["background-color: #cfe3f7"] * len(row)
    if key == restricted:
        return ["background-color: #f7ddc9"] * len(row)
    return [""] * len(row)

display(
    table.style.apply(highlight, axis=1)
    .format({"posterior mass": "{:.4f}", "best single path in group": "{:.4f}"})
    .hide(axis="index")
    .set_caption("blue = marginal MAP,  orange = Viterbi restriction")
)

ratio = masses[tuple(mm_path)] / masses[tuple(restricted)]
print(f"The marginal-MAP group carries {ratio:.2f}x the posterior mass of the "
      f"Viterbi restriction,")
print(f"despite containing a worse best-single-path "
      f"({best_in_group[tuple(mm_path)]:.4f} vs {best_in_group[tuple(restricted)]:.4f}).")

Read the two highlighted rows against each other. The orange group contains the
best single path in the table — it is the Viterbi optimum — and yet the blue
group carries more than twice its posterior mass, because it pools many
individually-worse paths. Marginal MAP picks the blue one; restricting Viterbi
picks the orange one.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

labels = [" ".join(k) for k, _ in ranked[:12]]
values = [v / total for _, v in ranked[:12]]
colors = [
    "#2E6DB4" if list(k) == mm_path
    else "#E08A3C" if list(k) == restricted
    else "0.75"
    for k, _ in ranked[:12]
]

bars = ax.bar(range(len(values)), values, color=colors)
ax.set_xticks(range(len(values)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("posterior mass")
ax.set_xlabel(f"assignment at t = {QUERY}")
ax.set_title(
    "Posterior mass per query-time assignment\n"
    "blue = marginal MAP (heaviest group)    orange = Viterbi restriction",
    loc="left", fontsize=11,
)
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)

fig.tight_layout()
plt.show()

## 4. Full coverage reduces to Viterbi

If every time is queried there is nothing left to sum over, and the two
algorithms must agree exactly. They do — and the marginal-MAP entry point warns,
because Viterbi computes the same answer more cheaply.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    all_times = marginal_map_torch_mvr_chmm(
        model, observed, query_times=list(range(T)),
        return_augmented=False, return_score=True,
    )

print(f"viterbi      : {' '.join(viterbi_path)}   {viterbi_ll:.6f}")
print(f"marginal MAP : {' '.join(all_times[0])}   {all_times[1]:.6f}")
print()
print("identical:", all_times[0] == viterbi_path
      and abs(all_times[1] - viterbi_ll) < 1e-5)
print()
for w in caught:
    print(f"{w.category.__name__}: {w.message}")

## 5. Adding a constraint

Constraints apply to the *whole* chain, including the summed-out times. A path
that violates an MVR at a time nobody asked about still contributes nothing to
any group's mass.

Here is the familiar "never visit `A`" MVR, checked against brute force with the
same acceptance test applied inside the enumeration.

In [ ]:
def forbid_state_mvr(forbidden_state, time_range=None):
    """MVR rejecting any path that visits `forbidden_state` inside its window."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("violated" if h == forbidden_state else "ok") for h in HIDDEN_STATES},
        upd={
            (m, h): ("violated" if m == "violated" or h == forbidden_state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        time_range=time_range,
    )


WINDOW = [3, 5]
mvr = forbid_state_mvr("A", time_range=WINDOW)
model_c = MVR_CHMM(hidden_markov_model=hmm, constraints=[mvr])

mm_c = marginal_map_torch_mvr_chmm(
    model_c, observed, query_times=QUERY,
    return_augmented=False, return_score=True,
)
bf_c = brute_force_marginal_map(
    QUERY, accepts=lambda p: "A" not in p[WINDOW[0]:WINDOW[1] + 1]
)

print(f"marginal MAP : {' '.join(mm_c[0])}   log-mass = {mm_c[1]:.4f}")
print(f"brute force  : {' '.join(bf_c[0])}   log-mass = {bf_c[1]:.4f}")
print()
print("agree:", mm_c[0] == bf_c[0] and abs(mm_c[1] - bf_c[1]) < 1e-4)
print()
inside = [t for t in range(WINDOW[0], WINDOW[1] + 1)]
summed = [t for t in inside if t not in QUERY]
print(f"The window {WINDOW} covers t = {inside}, of which {summed} are summed out.")
print("The constraint still binds there -- summed-out times are constrained too.")

Note which times the window covers. `t = 4` and `t = 5` are summed out, yet the
constraint is enforced on them. This is the case that forces the algorithm to
build a genuine transfer operator over the augmented (hidden × mediation) space:
the automaton's state has to be carried across the summed-out region, so the
mediation axis cannot be dropped there.

That operator costs $(K\cdot\prod_i M_i)^2$ memory. Aligning an MVR's
`time_range` to query times avoids it — a gap with no MVR active costs only
$K^2$.

## 6. Sparse observations and a long horizon

`marginal_map_torch_mvr_chmm` takes the same `observed` / `time_horizon`
arguments as the Viterbi entry point, so the query times, the observed times and
the horizon are three independent things.

This is where the algorithm earns its keep. When a stretch of time has no
observation and no active MVR, it collapses into a **power of the transition
matrix** instead of being stepped through one time at a time.

In [ ]:
BIG_T = 10_000
sparse = {0: "mid", BIG_T // 2: "hi", BIG_T - 1: "lo"}
big_query = [0, BIG_T // 2, BIG_T - 1]

t0 = time.perf_counter()
big = marginal_map_torch_mvr_chmm(
    model, sparse, time_horizon=BIG_T, query_times=big_query,
    return_augmented=False, return_score=True,
)
mm_time = time.perf_counter() - t0

t0 = time.perf_counter()
viterbi_torch_mvr_chmm(model, sparse, time_horizon=BIG_T, return_augmented=False)
vit_time = time.perf_counter() - t0

print(f"horizon      : {BIG_T} steps")
print(f"observed at  : {sorted(sparse)}")
print(f"queried at   : {big_query}")
print()
print(f"marginal MAP : {' '.join(big[0])}   log-mass = {big[1]:.4f}"
      f"   [{mm_time * 1000:.1f} ms]")
print(f"full Viterbi over all {BIG_T} steps          [{vit_time * 1000:.1f} ms]")
print()
print(f"speedup: {vit_time / mm_time:.0f}x")

Two gaps of ~5000 steps each, each collapsed into one `matrix_power` call
instead of 5000 sequential updates. The shortcut only applies when the gap has
no mediation axis and no interior observation; otherwise the interior is
propagated step by step, which is still correct, just slower.

## 7. Reading the augmented output

`return_augmented=True` reports one entry per **query time**, each tagged with
the global time it refers to, plus the mediation state of every MVR active
there.

In [ ]:
path_aug, augmented = marginal_map_torch_mvr_chmm(
    model_c, observed, query_times=QUERY
)

display(
    pd.DataFrame(
        [
            {
                "time": e["time"],
                "hidden": hmm.hidden_to_external[e["hidden_index"]],
                "MVR 0 mediation": e["mvr_states"].get(0, "-- inactive --"),
            }
            for e in augmented
        ]
    ).style.hide(axis="index").set_caption(
        f"Query times only.  MVR 0 is active on {WINDOW}."
    )
)

The MVR is active on `[3, 5]`, so it shows a mediation state at `t = 3` and
nothing at `t = 0` or `t = 6`. The rows are the query times, not the horizon —
the summed-out times have no single value to report.

## Notes

- **Marginal MAP is not Viterbi.** The states returned at the query times are in
  general not the states the joint MAP path takes there. Use
  `viterbi_torch_mvr_chmm` when the whole path is what you want.
- **The mediation state is maximized, not summed out.** The query is marginal MAP
  over the augmented chain, so a query time commits to a $(hidden, mediation)$
  pair and the reported hidden path is the projection of it. Two consequences:
  the enumeration in §2 must group by the augmented state in general (see the
  caveat there), and this does **not** agree with the per-time posterior —
  `forward_backward_mvr_chmm` sums the mediation out of `gamma`, because a
  marginal is a sum, so `argmax_h gamma[t]` and `marginal_map(query_times=[t])`
  can name different states. Both are right for the question they answer.
- Summed-out times remain part of the chain: they consume a transition, emit if
  observed, and are constrained by every MVR active there.
- The query is tractable on a chain because eliminating the summed-out run
  between two adjacent query times leaves a factor over just those two, so the
  induced width stays at 2. Marginal MAP is NP-hard on general graphs.
- **Everything runs in log space**, the sum-product gap eliminations as much as
  the max-plus recursion over the query times. A gap operator entry is the
  probability of crossing the gap *while satisfying every MVR active across it*,
  which decays exponentially in the gap length; in float32 probability space that
  saturates silently at about $-104$, the log of the smallest subnormal, rather
  than raising.
- The one exception is a **bare** gap — no MVR active and nothing observed inside
  — which collapses to a `matrix_power` of the transition matrix in probability
  space. That is safe precisely because a stochastic matrix stays stochastic under
  powering, so its rows cannot underflow.
- An MVR window spanning a summed-out run forces an explicit
  $(K\cdot\prod_i M_i)^2$ transfer operator. Aligning `time_range` to query
  times keeps gaps at $K^2$.
- `query_times=None` covers the horizon, reproduces Viterbi exactly, and warns.